In [0]:
-- Latest failed jobs
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_failures_recent AS
select account_id,workspace_id,job_id,job_name,run_id,run_type,trigger_type,result_state,period_start_time as job_start_time,period_end_time as job_end_time,termination_type FROM
(select * , ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY period_start_time DESC) AS rn FROM data_governance.gold_lineage.fact_job_performance where result_state!='SUCCEEDED')t
where rn<=5 ;

In [0]:
-- Queries with high execution time and multiple runs

WITH aggregated AS (
    SELECT
        statement_text,
        COUNT(*) AS execution_count,
        ROUND(AVG(query_duration_sec), 2) AS avg_duration_sec,
        MAX(query_duration_sec) AS max_duration_sec,
        SUM(read_mb) AS total_read_mb
    FROM data_governance.silver_lineage_analysis.lineage_query_history
    GROUP BY statement_text
    HAVING COUNT(*) > 1
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY avg_duration_sec DESC) AS rn
    FROM aggregated
),
sample_query AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY statement_text ORDER BY query_duration_sec DESC) AS rnk
        FROM data_governance.silver_lineage_analysis.lineage_query_history
    ) t
    WHERE rnk = 1
)
SELECT
    s.account_id,
    s.workspace_id,
    s.executed_by AS executor_id,
    s.execution_status,
    s.statement_id AS query_statement_id,
    s.statement_text,
    s.statement_type,
    r.execution_count,
    r.avg_duration_sec,
    r.max_duration_sec,
    r.total_read_mb
FROM ranked r
JOIN sample_query s
ON r.statement_text = s.statement_text
WHERE r.rn <= 10;

In [0]:
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_user_resource_usage AS
SELECT
    executed_by,
    SUM(read_mb) AS total_read_mb,
    SUM(written_mb) AS total_written_mb,
    SUM(shuffle_read_mb) AS total_shuffle_mb
FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY executed_by;

In [0]:
SELECT
    job_id,
    COUNT(*) AS total_queries,
    AVG(query_duration_sec) AS avg_duration_sec
FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY job_id;

In [0]:
SELECT
    event_date,
    COUNT(*) AS total_queries,
    COUNT(CASE WHEN execution_status = 'FINISHED' THEN 1 END) AS success_count,
    COUNT(CASE WHEN execution_status != 'FINISHED' THEN 1 END) AS failure_count,
    ROUND(AVG(query_duration_sec), 2) AS avg_duration_sec,
    ROUND(SUM(read_mb), 2) AS total_read_mb,
    ROUND(SUM(written_mb), 2) AS total_written_mb

FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY event_date;